# Async in python

## Coroutine objects

In [12]:
import asyncio

In [2]:
async def fetch_data():
    print ("started")

In [3]:
fetch_data()

<coroutine object fetch_data at 0x000002B5D47C7100>

In [4]:
coro = fetch_data()

In [5]:
await coro

started


In [6]:
async def fetch_data():
    print("A")
    print("B")

In [7]:
coro = fetch_data() # wont be executed because creating the coroutine object does not start it.

print("C")

C


In [8]:
async def fetch_data():
    print("A")
    print("B")

coro = fetch_data() # wont be executed yet

print("C") # 1st

await coro # 2nd

print("D") # 3rd

C
A
B
D


C:\Users\hamza\AppData\Local\Temp\ipykernel_20716\4187357328.py:5: RuntimeWarning: coroutine 'fetch_data' was never awaited
  coro = fetch_data() # wont be executed yet


In [9]:
async def some_io(name, delay):
    print(f"{name}: starting I/O")
    await asyncio.sleep(delay)
    print(f"{name}: I/O completed")

async def A ():
    print ("A1")
    await some_io("A", 3)
    print("A2")

async def B ():
    print ("B1")
    await some_io("B", 1)
    print("B2")
# Note we can have both A and B in one thread

In [14]:
coro_A  = A() # coroutine objects means "run A and wait until its finished"
coro_B  = B() # after A finished "now run B"

C:\Users\hamza\AppData\Local\Temp\ipykernel_20716\2770136778.py:2: RuntimeWarning: coroutine 'B' was never awaited
  coro_B  = B() # after A finished "now run B"


In [ ]:
result_A  = await coro_A # The value returned by the coroutine
result_B  = await coro_B 

A1
A: starting I/O
A: I/O completed
A2
B1
B: starting I/O
B: I/O completed
B2


``` text
await A()
   │
   ├── A1
   ├── A starts I/O
   ├── A waits 3 seconds
   ├── A I/O completes
   └── A2
          │
          ↓
     await B()
          │
          ├── B1
          ├── B starts I/O
          ├── B waits 1 second
          ├── B I/O completes
          └── B2
```

## Scheduling 

In [ ]:
import asyncio

# task_A, task_b are awaitable 
task_A = asyncio.create_task(A()) # Means schedule A, it does not create a thread. It schedules the coroutine as a Task on the event loop.
task_B = asyncio.create_task(B()) # Schedule B as another Task on the event loop.

A1
A: starting I/O
B1
B: starting I/O
B: I/O completed
B2
A: I/O completed
A2


In [17]:
type(task_A)

_asyncio.Task

``` text
Event Loop
│
├── Task A → running/waiting
└── Task B → running/waiting
```
``` text
A1
A: starting I/O
      ↓
A waits 3s
      ↓
B1
B: starting I/O
      ↓
B waits 1s
      ↓
B's I/O completes
      ↓
B2
      ↓
A's I/O completes
      ↓
A2
```

## Event loop

In [ ]:
async def main():
    print("Hello")

In [ ]:
main() # create a coroutine object, nothing drives it.

<coroutine object main at 0x000001EE057BB340>

In [31]:
# asyncio.run(main()) # wont run in Jupiter as it has already running loop, but it is essential for starting event loop in .py

In [ ]:
async def A():
    print("A1")
    await asyncio.sleep(2)
    print("A2")
async def main():
    task = asyncio.create_task(A())

In [ ]:
await main() # In .py file: await main()

A1


A2


In [ ]:
async def A():
    print ("A1")
    await asyncio.sleep(3)
    print ("A2")

async def main():
    task = asyncio.create_task(A())

    print ("B")

    await task # here `main` says: I need task A to finish before I can continue 

    print ("B")

In [25]:
await main()

B
A1
A2
B


## asyncio.gather()

In [37]:
async def A():
    print ("A")

async def B():
    print ("B")

async def C():
    print ("C")

In [38]:
task_A = asyncio.create_task(A())
task_B = asyncio.create_task(B())


A
B


In [39]:
await task_A
await task_B

In [ ]:
results = await asyncio.gather( # means "Run these awaitables concurrently and give me their results when they have all completed"
    A(), 
    B(), 
    C() 
)

A
B
C


In [49]:
async def A():
    await asyncio.sleep(3)
    print("A")
    return "A"

async def B():
    await asyncio.sleep(1)
    print("B")
    return "B"

async def C():
    await asyncio.sleep(2)
    print("C")
    return "C"

In [50]:
results = await asyncio.gather(  
    A(), 
    B(), 
    C() 
)


B
C
A


In [51]:
print(results)

['A', 'B', 'C']
